# Meme Decoder Colab Pipeline

This notebook runs the MemeCap ablation pipeline on Google Colab:

- Input settings `1-5`
- Training strategies: `zero-shot`, `projector-only`, `projector-lora`
- Google Drive data/cache/output paths

Recommended first run: use the smoke-test cells with small sample counts before launching full training.

## 1. Runtime And Google Drive

In Colab, choose **Runtime -> Change runtime type -> GPU -> A100**. The commands below are tuned for A100: higher image budget (`--max-pixels 1048576`), no 4-bit loading by default, and LoRA rank 16.

Then run the next cell to mount Google Drive. The notebook assumes your dataset is already uploaded under `/content/drive/MyDrive/meme-decoder-data/`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Basic GPU check
!nvidia-smi


Thu Apr 30 09:46:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Or Update The Repository

In [3]:
import os
from pathlib import Path

REPO_URL = "https://github.com/baoyunfan0101/meme-decoder.git"
REPO_DIR = Path("/content/meme-decoder")

if REPO_DIR.exists():
    %cd /content/meme-decoder
    !git pull
else:
    %cd /content
    !git clone $REPO_URL
    %cd /content/meme-decoder

/content
Cloning into 'meme-decoder'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 58), reused 74 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 47.51 KiB | 11.88 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/meme-decoder


## 3. Install Dependencies

In [4]:
%cd /content/meme-decoder
!pip install -q -r requirements.txt

# Keep the VLM stack current. torchao>=0.16 is required by recent Transformers quantization paths.
!pip install -q -U transformers accelerate peft bitsandbytes 'torchao>=0.16.0'

# If Colab had an older torchao already imported, restart the runtime after this cell before training.


/content/meme-decoder
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 166.4 MB/s eta 0:00:00


## 4. Hugging Face Login

Store your Hugging Face token in **Colab Secrets** with the name `HF_TOKEN` before running this cell. Do not paste tokens into the notebook or commit them to GitHub.

In Colab: left sidebar -> key icon / Secrets -> add `HF_TOKEN` -> enable notebook access.


In [5]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print('Loaded HF_TOKEN from Colab Secrets and logged in to Hugging Face.')
else:
    print('HF_TOKEN not found in Colab Secrets. Public downloads may still work, but with lower rate limits.')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded HF_TOKEN from Colab Secrets and logged in to Hugging Face.


## 5. Configure Data Paths

Expected data layout in Drive:

```text
/content/drive/MyDrive/meme-decoder-data/data/
  raw/
    memes-trainval.json
    memes-test.json
    memes/
      memes/
        memes/
          memes_bpet7l.png
          ...
  processed/
    memes-trainval.ocr.json
    memes-test.ocr.json
```

Set `RAW_DIR` to `data/raw`, not to the deepest image folder. `src/path_utils.py` automatically descends into nested `memes/` folders when locating images.

Because images are already uploaded to Drive, the training/evaluation commands below do **not** use `--allow-download`.


In [6]:
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/meme-decoder-data")
DATA_DIR = DRIVE_PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
IMAGE_CACHE_DIR = RAW_DIR / "memes" / "memes" / "memes"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["RAW_DIR"] = str(RAW_DIR)
os.environ["PROCESSED_DIR"] = str(PROCESSED_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("RAW_DIR        =", RAW_DIR)
print("IMAGE_CACHE_DIR=", IMAGE_CACHE_DIR)
print("PROCESSED_DIR  =", PROCESSED_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("raw exists      :", RAW_DIR.exists())
print("image dir exists:", IMAGE_CACHE_DIR.exists())
print("processed exists:", PROCESSED_DIR.exists())


RAW_DIR        = /content/drive/MyDrive/meme-decoder-data/data/raw
IMAGE_CACHE_DIR= /content/drive/MyDrive/meme-decoder-data/data/raw/memes/memes/memes
PROCESSED_DIR  = /content/drive/MyDrive/meme-decoder-data/data/processed
OUTPUT_DIR     = /content/drive/MyDrive/meme-decoder-data/outputs
raw exists      : True
image dir exists: True
processed exists: True


## 6. Verify Drive Data

This notebook expects the data to already be in Google Drive. It will not download the dataset by default.


In [7]:
required_paths = [
    RAW_DIR / "memes-trainval.json",
    RAW_DIR / "memes-test.json",
    PROCESSED_DIR / "memes-trainval.ocr.json",
    PROCESSED_DIR / "memes-test.ocr.json",
    IMAGE_CACHE_DIR,
]

for path in required_paths:
    print(f"{str(path):95s} exists={path.exists()}")

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Some required Drive paths are missing. Check the paths printed above.")


/content/drive/MyDrive/meme-decoder-data/data/raw/memes-trainval.json                           exists=True
/content/drive/MyDrive/meme-decoder-data/data/raw/memes-test.json                               exists=True
/content/drive/MyDrive/meme-decoder-data/data/processed/memes-trainval.ocr.json                 exists=True
/content/drive/MyDrive/meme-decoder-data/data/processed/memes-test.ocr.json                     exists=True
/content/drive/MyDrive/meme-decoder-data/data/raw/memes/memes/memes                             exists=True


## 7. Verify Dataset Counts


In [8]:
import json

for path in [PROCESSED_DIR / "memes-trainval.ocr.json", PROCESSED_DIR / "memes-test.ocr.json"]:
    print(path)
    if not path.exists():
        print("  MISSING")
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("  count:", len(data))
    print("  keys :", sorted(data[0].keys()) if data else [])

/content/drive/MyDrive/meme-decoder-data/data/processed/memes-trainval.ocr.json
  count: 5823
  keys : ['category', 'img_captions', 'img_fname', 'meme_captions', 'metaphors', 'ocr_confidence', 'ocr_engine', 'ocr_text', 'post_id', 'title', 'url']
/content/drive/MyDrive/meme-decoder-data/data/processed/memes-test.ocr.json
  count: 559
  keys : ['category', 'img_captions', 'img_fname', 'meme_captions', 'metaphors', 'ocr_confidence', 'ocr_engine', 'ocr_text', 'post_id', 'title', 'url']


## 8. Verify Local Meme Images

Images should already exist under `data/raw/memes/memes/memes/`. This check verifies that JSON `img_fname` values resolve to local files before running the model.

If many files are missing, fix the Drive upload path. URL download is available only as a commented fallback in the next cell.


In [9]:
import json
from pathlib import Path

def check_images(json_path, limit=20):
    records = json.loads(Path(json_path).read_text(encoding="utf-8"))
    missing = []
    checked = 0
    for record in records[:limit]:
        fname = record.get("img_fname", "")
        path = IMAGE_CACHE_DIR / fname
        checked += 1
        if not path.exists():
            missing.append(str(path))
    print(f"Checked {checked} examples from {Path(json_path).name}")
    print(f"Missing: {len(missing)}")
    for path in missing[:10]:
        print("  missing:", path)

check_images(PROCESSED_DIR / "memes-trainval.ocr.json", limit=20)
check_images(PROCESSED_DIR / "memes-test.ocr.json", limit=20)


Checked 20 examples from memes-trainval.ocr.json
Missing: 0
Checked 20 examples from memes-test.ocr.json
Missing: 0


In [ ]:
# Optional fallback only. Do not run this if images are already uploaded to Drive.
# It downloads missing images from each record's URL and caches them under RAW_DIR.
#
# %cd /content/meme-decoder
# !python -m scripts.download_images --processed-dir "$PROCESSED_DIR" --raw-dir "$RAW_DIR"


## 8. Zero-Shot Smoke Test

Input setting `4` means: image + title + image captions + OCR.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy zero-shot \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --max-new-tokens 64 \
  --max-pixels 1048576 \
  --eval-max-samples 20


## 9. Projector-Only Smoke Training

This is a true smoke test: it trains on only `--train-max-samples 100` examples and validates on `--eval-max-samples 20`. Use it only to verify that training, checkpointing, and validation work.


In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 1048576 \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --train-max-samples 100 \
  --eval-max-samples 20


## 10. Projector + LoRA Smoke Training

This is a true smoke test: it trains on only `--train-max-samples 100` examples. For A100, this keeps projector + LoRA training in full precision, uses gradient checkpointing, lowers the image budget to `--max-pixels 524288`, and uses `--lora-r 8` for stability.


In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --gradient-checkpointing \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --lora-r 8 \
  --train-max-samples 100 \
  --eval-max-samples 20


## 11. Full Training Examples

Run these after the smoke tests pass.

In [10]:
# Full projector-only training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 1048576 \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 4


/content/meme-decoder
Running command:
/usr/bin/python3 -m scripts.train --input-setting meme_title_imgcap_ocr --strategy projector-only --model-name Qwen/Qwen2.5-VL-3B-Instruct --batch-size 1 --epochs 2 --lr 2e-05 --weight-decay 0.01 --grad-accum-steps 4 --max-new-tokens 64 --lora-r 16 --lora-alpha 32 --lora-dropout 0.05 --max-pixels 1048576 --train-folds 1 2 3 4 --val-fold 5 --fold-prefix memes --fold-suffix .ocr.json
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
preprocessor_config.json: 100% 350/350 [00:00<00:00, 1.36MB/s]
chat_template.json: 1.05kB [00:00, 1.66MB/s]
config.json: 1.37kB [00:00, 3.88MB/s]
tokenizer_config.json: 5.70kB [00:00, 11.9MB/s]
vocab.json: 2.78MB [00:00, 24.4MB/s]
merges.txt: 1.67MB [00:00, 23.7MB/s]
tokenizer.json: 7.03MB [00:00, 125MB/s]
model.safetensors.index.json: 65.4kB [00:00, 136MB/s]
Fetching 2 files: 100% 2/2 [00:20<00:00, 10.27s/it]
Download complete: 100% 7.51G/7.51G [

In [11]:
# Full projector + LoRA training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --gradient-checkpointing \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --lora-r 8


/content/meme-decoder
Running command:
/usr/bin/python3 -m scripts.train --input-setting meme_title_imgcap_ocr --strategy projector-lora --model-name Qwen/Qwen2.5-VL-3B-Instruct --batch-size 1 --epochs 2 --lr 2e-05 --weight-decay 0.01 --grad-accum-steps 4 --max-new-tokens 64 --lora-r 8 --lora-alpha 32 --lora-dropout 0.05 --max-pixels 524288 --gradient-checkpointing --train-folds 1 2 3 4 --val-fold 5 --fold-prefix memes --fold-suffix .ocr.json
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100% 824/824 [00:23<00:00, 35.50it/s]
Training strategy: projector-lora
Processor max_pixels: 524288
Gradient checkpointing: enabled
Unfrozen projector-related modules:
  - model.visual.merger.ln_q
  - model.visual.merger.mlp.0
  - model.visual.merger.mlp.2
Trainable params: 808,791,240 / 4,566,481,640 (0.177115)
PROJECT_ROOT: /content/meme-decoder
Train files: ['/content/drive/MyDrive/meme-decoder-data/data

## 12. Evaluate A Trained Checkpoint

Use the checkpoint list cell to find the run name, then set `MODEL_PATH`.

In [12]:
from pathlib import Path

ckpt_root = OUTPUT_DIR / "checkpoints"
if ckpt_root.exists():
    for path in sorted(ckpt_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)[:10]:
        print(path)
else:
    print("No checkpoints yet:", ckpt_root)

/content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/meme_title_imgcap_ocr_trainF1-2-3-4_valF5_Qwen2.5-VL-3B-Instruct_ce_20260430_130246_409736
/content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/meme_title_imgcap_ocr_trainF1-2-3-4_valF5_Qwen2.5-VL-3B-Instruct_ce_20260430_094744_567866
/content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/meme_title_imgcap_ocr_trainF1-2-3-4_valF5_Qwen2.5-VL-3B-Instruct_ce_20260430_094025_411743
/content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/meme_title_imgcap_ocr_trainF1-2-3-4_valF5_Qwen2.5-VL-3B-Instruct_ce_20260430_093424_810689
/content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/meme_title_imgcap_ocr_trainF1-2-3-4_valF5_Qwen2.5-VL-3B-Instruct_ce_20260430_092305_767844


In [13]:
# Change this to one of the paths printed above.
MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "<run_name>")
STRATEGY = "projector-lora"  # or "projector-only"
INPUT_SETTING = 4

%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy "$STRATEGY" \
  --input-setting "$INPUT_SETTING" \
  --model-path "$MODEL_PATH" \
  --checkpoint-name best \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --max-new-tokens 64 \
  --max-pixels 1048576


/content/meme-decoder
Running command:
/usr/bin/python3 -m scripts.evaluate --input-setting meme_title_imgcap_ocr --strategy projector-lora --model-name Qwen/Qwen2.5-VL-3B-Instruct --eval-json memes-test.ocr.json --max-new-tokens 64 --max-pixels 1048576 --model-path /content/drive/MyDrive/meme-decoder-data/outputs/checkpoints/<run_name> --checkpoint-name best
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/meme-decoder/scripts/evaluate.py", line 227, in <module>
    main()
  File "/content/meme-decoder/scripts/evaluate.py", line 120, in main
    checkpoint_path = resolve_model_checkpoint(args.model_path, args.checkpoint_name)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/meme-decoder/scripts/evaluate

## 13. Optional: Zero-Shot Input Ablation 1-5

In [ ]:
%cd /content/meme-decoder
for setting in [1, 2, 3, 4, 5]:
    print("\n=== zero-shot setting", setting, "===")
    !python -m scripts.run_pipeline \
      --mode eval \
      --strategy zero-shot \
      --input-setting "$setting" \
      --processed-dir "$PROCESSED_DIR" \
      --raw-dir "$RAW_DIR" \
      --output-dir "$OUTPUT_DIR" \
      --eval-json memes-test.ocr.json \
      --max-new-tokens 64 \
      --max-pixels 1048576 \
      --eval-max-samples 50
